## **Preprocessing of Dataset 3**


In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\nghui\FYP\EBT-main\data\Dataset 3\detailed_lables.csv")

df["Participant"] = df["Participant"].astype(int)

df_filtered = df[
    (df["Participant"] >= 300) &
    (df["Participant"] <= 492)
]

result = df_filtered[["Participant", "Depression_label"]]

print(result.head())

result.to_csv("participant_300_492_depression.csv", index=False)


   Participant  Depression_label
0          300                 0
1          301                 0
2          302                 0
3          303                 0
4          304                 0


In [2]:
df = pd.read_csv(r"participant_300_492_depression.csv")

num_depression = (df["Depression_label"] == 1).sum()
num_non_depression = (df["Depression_label"] == 0).sum()

print("Depression:", num_depression)
print("Non-depression:", num_non_depression)


Depression: 42
Non-depression: 147


In [3]:
import os
import zipfile
import tarfile
import shutil

data_dir = r"C:\Users\nghui\FYP\EBT-main\data\Dataset 3\Data"
output_dir = r"C:\Users\nghui\FYP\EBT-main\data\Dataset 3\transcripts"

os.makedirs(output_dir, exist_ok=True)

def save_file(src_path, base_name):
    dst = os.path.join(output_dir, base_name + "_transcript.csv")
    shutil.move(src_path, dst)

for fname in os.listdir(data_dir):
    fpath = os.path.join(data_dir, fname)

    # -------- ZIP --------
    if fname.endswith(".zip"):
        with zipfile.ZipFile(fpath, "r") as z:
            for member in z.namelist():
                if "transcript" in member.lower() and member.endswith(".csv"):
                    z.extract(member, data_dir)
                    src = os.path.join(data_dir, member)
                    save_file(src, fname.replace(".zip", ""))




In [4]:
import os
import zipfile
import tarfile
import shutil
import re

data_dir = r"C:\Users\nghui\FYP\EBT-main\data\Dataset 3\Data2"
output_dir = r"C:\Users\nghui\FYP\EBT-main\data\Dataset 3\transcripts"

os.makedirs(output_dir, exist_ok=True)

bad_files = []         
processed_files = []  

def save_file(src_path, base_name):
    dst = os.path.join(output_dir, base_name + "_transcript.csv")
    shutil.move(src_path, dst)

for fname in os.listdir(data_dir):
    fpath = os.path.join(data_dir, fname)

    match = re.match(r"(\d+)_", fname)
    if not match:
        continue

    subject_id = int(match.group(1))
    if fname.endswith(".zip"):
        try:
            with zipfile.ZipFile(fpath, "r") as z:
                for member in z.namelist():
                    if re.match(r"^\d+_TRANSCRIPT\.csv$", os.path.basename(member)):
                        z.extract(member, data_dir)
                        src = os.path.join(data_dir, member)
                        save_file(src, fname.replace(".zip", ""))
                        processed_files.append(fname)
                        break

        except zipfile.BadZipFile:
            bad_files.append(fname)


print(f"Processed files: {len(processed_files)}")
print(f"Bad files: {len(bad_files)}")

print("\nBad file list:")
for f in bad_files:
    print(f)


Processed files: 189
Bad files: 0

Bad file list:


In [5]:
import os
import pandas as pd

CSV_DIR = r"C:\Users\nghui\FYP\EBT-main\data\Dataset 3\transcripts"

format_map = {}
format_id = 1

for file in sorted(os.listdir(CSV_DIR)):
    if file.lower().endswith(".csv"):
        path = os.path.join(CSV_DIR, file)
        
        try:
            df = pd.read_csv(path)
            cols = tuple(df.columns.tolist())  
            
            if cols not in format_map:
                format_map[cols] = {
                    "id": format_id,
                    "files": []
                }
                format_id += 1
            
            format_map[cols]["files"].append(file)
        
        except Exception as e:
            print(f"[ERROR] {file}: {e}")

for cols, info in format_map.items():
    print(f"\n=== Format {info['id']} ===")
    print("Columns:", cols)
    print("Files:")
    for f in info["files"]:
        print(" -", f)



=== Format 1 ===
Columns: ('start_time\tstop_time\tspeaker\tvalue',)
Files:
 - 300_P_transcript.csv
 - 301_P_transcript.csv
 - 302_P_transcript.csv
 - 303_P_transcript.csv
 - 304_P_transcript.csv
 - 305_P_transcript.csv
 - 306_P_transcript.csv
 - 307_P_transcript.csv
 - 308_P_transcript.csv
 - 309_P_transcript.csv
 - 310_P_transcript.csv
 - 311_P_transcript.csv
 - 312_P_transcript.csv
 - 313_P_transcript.csv
 - 314_P_transcript.csv
 - 315_P_transcript.csv
 - 316_P_transcript.csv
 - 317_P_transcript.csv
 - 318_P_transcript.csv
 - 319_P_transcript.csv
 - 320_P_transcript.csv
 - 321_P_transcript.csv
 - 322_P_transcript.csv
 - 323_P_transcript.csv
 - 324_P_transcript.csv
 - 325_P_transcript.csv
 - 326_P_transcript.csv
 - 327_P_transcript.csv
 - 328_P_transcript.csv
 - 329_P_transcript.csv
 - 330_P_transcript.csv
 - 331_P_transcript.csv
 - 332_P_transcript.csv
 - 333_P_transcript.csv
 - 334_P_transcript.csv
 - 335_P_transcript.csv
 - 336_P_transcript.csv
 - 337_P_transcript.csv
 - 338_P_tr

In [6]:
import pandas as pd
import re

def extract_text_after_speaker(line):
    parts = re.split(r"\s+", line.strip())

    if len(parts) < 4:
        return None

    speaker = parts[2].lower()

    if speaker != "participant":
        return None

    return " ".join(parts[3:])


def load_and_extract_text(csv_path):
    df = pd.read_csv(csv_path, sep=None, engine="python", header=0)

    if "Text" in df.columns:
        texts = df["Text"]

    elif "value" in df.columns and "speaker" in df.columns:
        df = df[df["speaker"].str.lower() == "participant"]
        texts = df["value"]

    else:
        return pd.DataFrame(columns=["text"])

    texts = (
        texts.dropna()
        .astype(str)
        .str.lower()
        .str.replace(r"<[^>]+>", " ", regex=True)   # remove DAIC tags
        .str.replace(r"\bxxx\b", " ", regex=True)   # remove unknown words
        .str.replace(r"\s+", " ", regex=True)       # normalize spaces
        .str.strip()
    )

    texts = texts[texts != ""]

    return texts.to_frame(name="text")




import os

INPUT_DIR = r"C:\Users\nghui\FYP\EBT-main\data\Dataset 3\transcripts"
OUTPUT_DIR = r"C:\Users\nghui\FYP\EBT-main\data\Dataset 3\Clean_Text"

os.makedirs(OUTPUT_DIR, exist_ok=True)

for file in os.listdir(INPUT_DIR):
    if file.lower().endswith(".csv"):
        input_path = os.path.join(INPUT_DIR, file)

        df_clean = load_and_extract_text(input_path)

        if len(df_clean) == 0:
            print(f"[WARNING] Empty after cleaning: {file}")
            continue

        output_path = os.path.join(OUTPUT_DIR, file)
        df_clean.to_csv(output_path, index=False)

print("Files saved")


Files saved


In [7]:
import pandas as pd
import os
import shutil
from sklearn.model_selection import train_test_split

LABEL_FILE = r"C:\Users\nghui\FYP\EBT-main\data\Dataset 3\Labels.csv"
CLEAN_TEXT_DIR = r"C:\Users\nghui\FYP\EBT-main\data\Dataset 3\Clean_Text"
OUTPUT_BASE = r"C:\Users\nghui\FYP\EBT-main\data\Dataset 3\Split"

os.makedirs(OUTPUT_BASE, exist_ok=True)
df = pd.read_csv(LABEL_FILE)

df["Participant"] = df["Participant"].astype(str)
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["Depression_label"], 
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["Depression_label"],
    random_state=42
)
splits = {
    "train": train_df,
    "val": val_df,
    "test": test_df
}

for split in splits:
    os.makedirs(os.path.join(OUTPUT_BASE, split, "samples"), exist_ok=True)
for split_name, split_df in splits.items():
    split_dir = os.path.join(OUTPUT_BASE, split_name)
    sample_dir = os.path.join(split_dir, "samples")

    split_df.to_csv(os.path.join(split_dir, "labels.csv"), index=False)

    for pid in split_df["Participant"]:
        filename = f"{pid}_P_TRANSCRIPT.csv"
        src = os.path.join(CLEAN_TEXT_DIR, filename)

        if os.path.exists(src):
            shutil.copy(src, os.path.join(sample_dir, filename))
        else:
            print(f"[WARNING] Missing sample for participant {pid}")
print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

print("\nClass distribution:")
print("Train:\n", train_df["Depression_label"].value_counts())
print("Val:\n", val_df["Depression_label"].value_counts())
print("Test:\n", test_df["Depression_label"].value_counts())


Train: 132
Val  : 28
Test : 29

Class distribution:
Train:
 Depression_label
0    103
1     29
Name: count, dtype: int64
Val:
 Depression_label
0    22
1     6
Name: count, dtype: int64
Test:
 Depression_label
0    22
1     7
Name: count, dtype: int64
